In [1489]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.optimizers import Adam
import gym
import pandas as pd

In [1504]:
# Define the PPO agent
class PPOAgent:
    def __init__(self, state_dim, action_dim, hidden_dim=64, lr=0.001, clip_ratio=0.2):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.clip_ratio = clip_ratio
        self.model = self._build_model()
        self.optimizer = Adam(learning_rate=lr)

    def _build_model(self):
        state_input = Input(shape=(self.state_dim,))
        x = Dense(self.hidden_dim, activation='relu')(state_input)
        x = Dense(self.hidden_dim, activation='relu')(x)
        mu = Dense(self.action_dim, activation='tanh')(x)
        sigma = Dense(self.action_dim, activation='softplus')(x)
        model = Model(state_input, [mu, sigma])
        return model

    def get_action(self, state):
        state = np.reshape(state, [1, self.state_dim])
        mu, sigma = self.model.predict(state)
        action = np.random.normal(mu[0], sigma[0], size=self.action_dim)
        return action

    def compute_loss(self, states, actions, old_probs, advantages):
        mu, sigma = self.model(states)
        new_probs = tf.reduce_sum(tf.exp(-0.5 * ((actions - mu) / sigma) ** 2) / (sigma * tf.sqrt(2 * np.pi)), axis=1)
        ratio = new_probs / old_probs
        clipped_ratio = tf.clip_by_value(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio)
        surrogate_loss = -tf.reduce_mean(tf.minimum(ratio * advantages, clipped_ratio * advantages))
        return surrogate_loss

    def train(self, states, actions, old_probs, advantages):
        with tf.GradientTape() as tape:
            loss = self.compute_loss(states, actions, old_probs, advantages)
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

In [1505]:
# Define the MAML meta-learner
class MAML:
    def __init__(self, agent, meta_lr=0.001):
        self.agent = agent
        self.meta_lr = meta_lr
        self.meta_optimizer = Adam(learning_rate=meta_lr)

    def adapt(self, task_states, task_actions, task_old_probs, task_advantages):
        with tf.GradientTape() as tape:
            loss = self.agent.compute_loss(task_states, task_actions, task_old_probs, task_advantages)
        grads = tape.gradient(loss, self.agent.model.trainable_variables)
        adapted_vars = [v - self.meta_lr * g for v, g in zip(self.agent.model.trainable_variables, grads)]
        return adapted_vars

    def meta_train(self, tasks, batch_size=32, num_steps=5):
        meta_losses = []
        for task in tasks:
            task_states, task_actions, task_old_probs, task_advantages = task
            adapted_vars = self.adapt(task_states, task_actions, task_old_probs, task_advantages)
            with tf.GradientTape() as tape:
                loss = self.agent.compute_loss(task_states, task_actions, task_old_probs, task_advantages)
            grads = tape.gradient(loss, adapted_vars)
            print("Gradients:", grads)  # Print gradients to debug
            self.meta_optimizer.apply_gradients(zip(grads, self.agent.model.trainable_variables))
            meta_losses.append(loss)
        return tf.reduce_mean(meta_losses)

In [1506]:
# Define the environment
class StockTradingEnv(gym.Env):
    def __init__(self, data):
        self.data = data
        self.state_dim = 4  # open, high, low, close
        self.action_dim = 1
        self.current_step = 0
        self.max_steps = len(data) - 1
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(self.state_dim,), dtype=np.float32)
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.action_dim,), dtype=np.float32)

    def reset(self):
        self.current_step = 0
        return self._get_observation()

    def step(self, action):
        reward = self._get_reward(action)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        obs = self._get_observation()
        return obs, reward, done, {}

    def _get_observation(self):
        return self.data.iloc[self.current_step][['Open', 'High', 'Low', 'Close']].values

    def _get_reward(self, action):
        close_price = self.data.iloc[self.current_step]['Close']
        next_close_price = self.data.iloc[self.current_step + 1]['Close']
        return (next_close_price - close_price) * action[0]

In [1501]:
def load_data(filename):
    data = pd.read_csv(filename)
    data = data[['Open', 'High', 'Low', 'Close']]
    data = (data - data.mean()) / data.std()
    return data

In [1507]:
# Main training loop
def train(agent, maml, env, data, episodes=100, batch_size=32, num_steps=5):
    for episode in range(episodes):
        tasks = []
        for _ in range(batch_size):
            env.reset()
            states, actions, rewards, old_probs = [], [], [], []
            for step in range(num_steps):
                state = env.reset()
                action = agent.get_action(state)
                next_state, reward, done, _ = env.step(action)
                states.append(state)
                actions.append(action)
                rewards.append(reward)
                old_probs.append(1.0)  # Placeholder for old probabilities
            tasks.append((np.array(states), np.array(actions), np.array(old_probs), np.array(rewards)))
        meta_loss = maml.meta_train(tasks, batch_size, num_steps)
        print(f"Episode {episode + 1}/{episodes}, Meta Loss: {meta_loss}")

In [1508]:
# Example usage
if __name__ == "__main__":
    data = load_data('./datas/samsung.csv')
    env = StockTradingEnv(data)
    agent = PPOAgent(state_dim=env.state_dim, action_dim=env.action_dim)
    maml = MAML(agent)
    train(agent, maml, env, data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

ValueError: No gradients provided for any variable.